In [4]:
# Compact stationarity test function for time series data.

import warnings
import numpy as np

from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tools.sm_exceptions import InterpolationWarning


def stationarity_test(x):

    x = x.dropna()

    if x.nunique() <= 2:

        return {
            "adf_p": np.nan,
            "kpss_p": np.nan,
            "stationary": False
        }

    adf_p = adfuller(
        x,
        autolag="AIC"   # AIC is better but need to confirm the reason
    )[1]

    with warnings.catch_warnings():

        warnings.simplefilter(
            "ignore",
            InterpolationWarning
        )

        try:
            kpss_p = kpss(
                x,
                regression="c",
                nlags="auto"
            )[1]

        except:
            kpss_p = np.nan

    return {
        "adf_p": adf_p,
        "kpss_p": kpss_p,
        "stationary":
            (adf_p < 0.05)
            and
            (kpss_p > 0.05)
    }

In [10]:
def create_change_series(folder: Path):

    fixed_file = (
        folder /
        "gt_stitched_fixed_2022-01-01_2026-05-31.csv"
    )

    df = pd.read_csv(
        fixed_file,
        parse_dates=["Time"]
    )


    value_col = [
        c for c in df.columns
        if c != "Time"
    ][0]


    ts = (
        df
        .set_index("Time")[value_col]
        .sort_index()
        .astype(float)
    )


    # -----------------------------
    # Diagnostics
    # -----------------------------

    zero_share = (ts <= 0).mean()

    flat_share = (
        ts.diff()
        .fillna(0)
        .eq(0)
        .mean()
    )

    # -----------------------------
    # Create both transformations
    # -----------------------------

    diff = (
        ts.diff()
        .dropna()
    )

    diff.name = value_col


    log_diff = (
        np.log1p(ts)
        .diff()
        .dropna()
    )

    log_diff.name = value_col


    # -----------------------------
    # Weekend adjustment
    # Combine Friday + Saturday + Sunday
    # because S&P500 is closed on weekends
    # -----------------------------

    def friday_weekend_adjust(series):

        series = series.copy()

        for date in series.index:

            if date.dayofweek == 4:   # Friday

                weekend_dates = [
                    date,
                    date + pd.Timedelta(days=1),
                    date + pd.Timedelta(days=2)
                ]

                values = series.reindex(
                    weekend_dates
                )

                # cumulative weekend information
                series.loc[date] = (
                    values.mean()
                )


        # remove Saturday/Sunday
        series = series[
            ~series.index.dayofweek.isin([5, 6])
        ]


        return series


    diff = friday_weekend_adjust(
        diff
    )

    log_diff = friday_weekend_adjust(
        log_diff
    )



    # -----------------------------
    # Stationarity tests
    # -----------------------------

    level_test = stationarity_test(ts)

    diff_test = stationarity_test(
        diff
    )

    log_diff_test = stationarity_test(
        log_diff
    )



    # -----------------------------
    # Save both outputs
    # -----------------------------

    diff.reset_index().to_csv(
        folder /
        "gt_diff.csv",
        index=False
    )


    log_diff.reset_index().to_csv(
        folder /
        "gt_log_diff.csv",
        index=False
    )

    # -----------------------------
    # Diagnostic text
    # -----------------------------

    lines = [

        "Google Trends Transformation Diagnostic",
        "=" * 60,

        f"Term: {folder.name}",
        "",

        "Original series:",
        f"Observations       : {len(ts)}",
        f"Date range         : {ts.index.min().date()} → {ts.index.max().date()}",
        f"Missing values     : {ts.isna().sum()}",
        "",

        "Zero / flat behavior:",
        f"Zero share         : {zero_share:.4f}",
        f"Flat share         : {flat_share:.4f}",
        "",

        "Level stationarity:",
        str(level_test),
        "",

        "First difference stationarity:",
        str(diff_test),
        "",

        "Log difference stationarity:",
        str(log_diff_test),

    ]


    (folder /
     "transform_diagnostic.txt"
    ).write_text(
        "\n".join(lines),
        encoding="utf-8"
    )


    return {

        "term": folder.name,

        "observations": len(ts),

        "zero_share": round(
            zero_share,
            4
        ),

        "flat_share": round(
            flat_share,
            4
        ),

        "level_stationary":
            level_test["stationary"],

        "diff_stationary":
            diff_test["stationary"],

        "log_diff_stationary":
            log_diff_test["stationary"]

    }

In [11]:
from pathlib import Path
import pandas as pd


DATA_DIR = Path(
    r"C:\Python\CSUREMM\test"
)


stationarity_summary = []
failed = []


for folder in DATA_DIR.iterdir():

    if not folder.is_dir():
        continue

    try:

        result = create_change_series(
            folder
        )

        stationarity_summary.append(
            result
        )

        print(
            "done:",
            folder.name
        )


    except Exception as e:

        failed.append(
            (
                folder.name,
                str(e)
            )
        )

        print(
            "failed:",
            folder.name,
            e
        )


# -----------------------------
# Save batch summaries
# -----------------------------

summary = pd.DataFrame(
    stationarity_summary
)


summary.to_csv(
    DATA_DIR /
    "gt_stationarity_summary.csv",
    index=False
)


with open(
    DATA_DIR /
    "gt_stationarity_failures.txt",
    "w",
    encoding="utf-8"
) as f:

    for term, error in failed:

        f.write(
            f"{term}: {error}\n"
        )


print()
print(
    f"Completed: {len(summary)} terms"
)

print(
    f"Failed: {len(failed)} terms"
)

done: accrue
done: advantage
done: affluence
done: affluent
done: afloat
done: allowance
done: aristocracy
done: aristocrat
done: aristocratic
done: associate
done: backer
done: backward
done: backwardness
done: bankrupt
done: bankruptcy
done: bargain
done: beggar
done: benefactor
done: beneficiary
done: benefit
done: benevolence
done: benevolent
done: bequeath
done: betroth
done: betrothal
done: blackmail
done: bonus
done: boom
done: breadwinner
done: bribe
done: broke
done: bum
done: buy
done: capitalize
done: charitable
done: charity
done: cheap
done: colony
done: commoner
done: community
done: compensate
done: compensation
done: contribute
done: contribution
done: cooperative
done: corrupt
done: cost
done: costly
done: crisis
done: debtor
done: default
done: deficit
done: depreciation
done: depression
done: destitute
done: domination
done: donate
done: donation
done: economize
done: endow
done: entrepreneurial
done: equity
done: expense
done: expensive
done: extravagant
done: fello

In [12]:
# batch all words

def build_gt_panel(
    data_dir: Path,
    file_name: str,
    output_name: str
):

    series_list = []


    for folder in sorted(data_dir.iterdir()):

        if not folder.is_dir():
            continue


        file = folder / file_name


        if not file.exists():
            continue


        try:

            df = pd.read_csv(
                file,
                parse_dates=["Time"]
            )


            value_col = [
                c for c in df.columns
                if c != "Time"
            ][0]


            s = (
                df
                .set_index("Time")[value_col]
                .rename(folder.name)
            )


            series_list.append(s)


        except Exception as e:

            print(
                f"failed: {folder.name} | {e}"
            )


    if len(series_list) == 0:
        raise ValueError(
            f"No {file_name} files found"
        )


    panel = pd.concat(
        series_list,
        axis=1,
        join="outer"
    )


    panel = (
        panel
        .sort_index()
    )


    # remove duplicate dates if any
    panel = (
        panel
        .loc[
            ~panel.index.duplicated()
        ]
    )


    output = data_dir / output_name


    panel.to_csv(
        output,
        index=True
    )


    print(
        f"Saved {output_name}"
    )

    print(
        f"Terms: {panel.shape[1]}"
    )

    print(
        f"Days: {panel.shape[0]}"
    )

    print(
        f"Missing cells: {panel.isna().sum().sum()}"
    )


    return panel



# -----------------------------
# Build two GT master panels
# -----------------------------


gt_diff_panel = build_gt_panel(
    DATA_DIR,
    "gt_diff.csv",
    "gt_diff_panel.csv"
)


gt_logdiff_panel = build_gt_panel(
    DATA_DIR,
    "gt_log_diff.csv",
    "gt_log_diff_panel.csv"
)

Saved gt_diff_panel.csv
Terms: 150
Days: 1150
Missing cells: 0
Saved gt_log_diff_panel.csv
Terms: 150
Days: 1150
Missing cells: 0


In [13]:
# ------------------------------------
# Save master stationarity diagnosis
# ------------------------------------

stationarity_panel = pd.DataFrame(
    stationarity_summary
)


stationarity_panel = (
    stationarity_panel
    .sort_values(
        "term"
    )
    .reset_index(
        drop=True
    )
)


output_file = (
    DATA_DIR /
    "stationarity_summary.csv"
)


stationarity_panel.to_csv(
    output_file,
    index=False
)


print(
    f"Saved: {output_file}"
)


print(
    stationarity_panel.head()
)

Saved: C:\Python\CSUREMM\test\stationarity_summary.csv
        term  observations  zero_share  flat_share  level_stationary  \
0     accrue          1612      0.0031      0.0279             False   
1  advantage          1612      0.0000      0.0651             False   
2  affluence          1612      0.4119      0.2835             False   
3   affluent          1612      0.0000      0.0440             False   
4     afloat          1612      0.1253      0.0471              True   

   diff_stationary  log_diff_stationary  
0            False                 True  
1             True                 True  
2             True                 True  
3             True                 True  
4             True                 True  


** Start selecting words for indices **

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

from scipy.stats import pearsonr
from statsmodels.stats.multitest import multipletests

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV
from sklearn.pipeline import Pipeline


DATA_DIR = Path(
    r"C:\Python\CSUREMM\test"
)

GOOD_TERMS = (
    DATA_DIR /
    "good_terms.csv"
)

GT_FILE = (
    "gt_stationary_2022-01-02_2026-05-31.csv"
)

SP500_FILE = (
    DATA_DIR /
    "sp500_daily.csv"
)

In [13]:
def load_google_terms():

    terms = pd.read_csv(
        GOOD_TERMS
    )["term"].tolist()


    frames = []


    for term in terms:

        file = (
            DATA_DIR /
            term /
            GT_FILE
        )


        if not file.exists():
            continue


        df = pd.read_csv(
            file,
            parse_dates=["Time"]
        )


        value_col = (
            df.columns
            .drop("Time")[0]
        )


        df = df.rename(
            columns={
                value_col: term
            }
        )


        df = df.set_index(
            "Time"
        )


        frames.append(
            df[[term]]
        )


    gt = pd.concat(
        frames,
        axis=1
    )


    return gt



gt = load_google_terms()

print(
    gt.shape
)

gt.head()

(1611, 142)


,abundance,accrue,advantage,affluence,affluent,afloat,allowance,aristocracy,aristocrat,aristocratic,...,thrifty,treasure,underworld,unemployed,vagabond,vagrant,valuable,warfare,waste,worth
Time,,,,,,,,,,,,,,,,,,,,,
2022-01-02,-0.453341,0.376885,0.078637,0.0,0.279585,0.026840,0.275012,0.414112,-0.713598,0.000000,...,-0.142451,0.024681,-0.080127,0.094583,0.161036,0.000000,0.079749,0.041487,0.099659,0.010698
2022-01-03,-0.066650,0.529666,0.494421,0.0,0.571705,0.040751,0.285225,0.425768,0.560360,0.000000,...,0.178021,-0.109195,-0.132496,0.325307,-0.231442,0.064612,-0.082293,-0.166108,0.369702,-0.112022
2022-01-04,0.297808,0.108001,0.039103,0.0,-0.115583,-3.945901,0.179707,-0.370642,0.187887,0.000000,...,0.000000,0.083687,0.048744,0.072249,0.005434,0.060921,0.246043,0.147315,0.122368,0.105991
2022-01-05,-0.195939,-0.263217,-0.047434,0.0,0.017959,0.000000,-0.178313,0.061789,-0.166194,3.831635,...,0.000000,-0.031671,0.016656,0.147602,0.060698,-0.119648,-0.126456,0.011157,-0.052083,-0.070830
2022-01-06,-0.040088,-0.168974,-0.266053,0.0,-0.356988,3.842633,-0.248892,0.295340,-0.135242,-3.831635,...,-0.161718,-0.201608,-0.192007,-0.368965,-0.185217,0.064612,-0.195964,-0.283304,-0.432025,-0.244851


In [ ]:
gt.to_csv(
    "gt_good_terms.csv")

In [14]:
sp500 = pd.read_csv(
    SP500_FILE,
    parse_dates=["date"]
)


sp500 = (
    sp500
    .set_index("date")
)


target = (
    sp500["sp500_rv5"]
)


data = pd.concat(
    [
        gt,
        target
    ],
    axis=1
).dropna()


print(
    data.shape
)

(1100, 143)


In [18]:
from scipy.stats import pearsonr


def lag_corr_series(
    x,
    y,
    lag
):
    """
    x: Google Trends series
    y: volatility series

    lag > 0:
        x(t) predicts y(t+lag)
    """

    df = pd.concat(
        [
            x,
            y
        ],
        axis=1
    ).dropna()


    x = df.iloc[:,0]
    y = df.iloc[:,1]


    if lag > 0:

        x_lag = x.iloc[:-lag]
        y_lag = y.iloc[lag:]


    else:

        x_lag = x
        y_lag = y


    if len(x_lag) < 30:
        return np.nan, np.nan


    r,p = pearsonr(
        x_lag,
        y_lag
    )

    return r,p

In [20]:
def correlation_screen(
    gt,
    target
):

    results=[]

    lags=[
        0,
        1,
        3,
        5
    ]


    for term in gt.columns:


        for lag in lags:

            r,p = lag_corr_series(
                gt[term],
                target,
                lag
            )


            results.append(
                {
                    "term":term,
                    "lag":lag,
                    "corr":r,
                    "pvalue":p
                }
            )


    return pd.DataFrame(results)

In [21]:
corr_results = correlation_screen(
    gt,
    target
)

print(
    corr_results.shape
)

corr_results.head()

(568, 4)


,term,lag,corr,pvalue
0,abundance,0,-0.018370,0.542765
1,abundance,1,-0.026270,0.384274
2,abundance,3,-0.032890,0.276419
3,abundance,5,-0.027631,0.361006
4,accrue,0,-0.012821,0.671002


In [22]:
ranking = (
    corr_results
    .assign(
        abs_corr=lambda x:
        x["corr"].abs()
    )
    .sort_values(
        "abs_corr",
        ascending=False
    )
)


print(
    ranking
    .head(30)
    .to_string(
        index=False
    )
)

       term  lag      corr   pvalue  abs_corr
    subsidy    0 -0.084443 0.005071  0.084443
       ruin    0 -0.076063 0.011619  0.076063
    warfare    5  0.073377 0.015158  0.073377
  recession    5  0.069543 0.021370  0.069543
  allowance    0 -0.060995 0.043120  0.060995
       ruin    5 -0.059390 0.049444  0.059390
    savings    5 -0.059286 0.049843  0.059286
liquidation    3 -0.058960 0.050906  0.058960
       laid    3 -0.058780 0.051616  0.058780
    warfare    3  0.057375 0.057470  0.057375
 profitable    0 -0.057173 0.058010  0.057173
liquidation    0 -0.056920 0.059132  0.056920
   donation    0 -0.056880 0.059310  0.056880
   donation    3 -0.056735 0.060312  0.056735
  blackmail    0 -0.056004 0.063340  0.056004
liquidation    1 -0.055425 0.066251  0.055425
  blackmail    3 -0.054843 0.069410  0.054843
    savings    3 -0.054010 0.073753  0.054010
    warfare    0  0.053976 0.073542  0.053976
   donation    5 -0.053804 0.075131  0.053804
       cost    0 -0.053181 0.07789

In [39]:
# Keep the strongest lag for each term
best_rank = (
    ranking
    .groupby(
        "term",
        as_index=False
    )
    .first()
)


# Re-sort after groupby
best_rank = (
    best_rank
    .sort_values(
        "abs_corr",
        ascending=False
    )
)


print(
    best_rank.head(30).to_string(index=False)
)

           term  lag      corr   pvalue  abs_corr
        subsidy    0 -0.084443 0.005071  0.084443
           ruin    0 -0.076063 0.011619  0.076063
        warfare    5  0.073377 0.015158  0.073377
      recession    5  0.069543 0.021370  0.069543
      allowance    0 -0.060995 0.043120  0.060995
        savings    5 -0.059286 0.049843  0.059286
    liquidation    3 -0.058960 0.050906  0.058960
           laid    3 -0.058780 0.051616  0.058780
     profitable    0 -0.057173 0.058010  0.057173
       donation    0 -0.056880 0.059310  0.056880
      blackmail    0 -0.056004 0.063340  0.056004
           cost    0 -0.053181 0.077893  0.053181
    inexpensive    0 -0.052585 0.081289  0.052585
         reward    5 -0.049802 0.099535  0.049802
          guide    1 -0.047704 0.113982  0.047704
     generosity    3  0.047149 0.118594  0.047149
          bonus    0 -0.047050 0.118862  0.047050
         gamble    1  0.046351 0.124621  0.046351
         debtor    0 -0.045077 0.135151  0.045077


In [42]:
N = 50

selected_terms = (
    best_rank
    .head(N)
    ["term"]
    .tolist()
)


print(
    len(selected_terms),
    selected_terms
)

50 ['subsidy', 'ruin', 'warfare', 'recession', 'allowance', 'savings', 'liquidation', 'laid', 'profitable', 'donation', 'blackmail', 'cost', 'inexpensive', 'reward', 'guide', 'generosity', 'bonus', 'gamble', 'debtor', 'thrifty', 'rich', 'entrepreneurial', 'tariff', 'partner', 'luxury', 'fellowship', 'charity', 'underworld', 'richness', 'bribe', 'depreciation', 'bankrupt', 'steal', 'prosper', 'skill', 'cheap', 'owe', 'race', 'charitable', 'nobility', 'subsidize', 'deficit', 'equity', 'buy', 'expense', 'backer', 'partnership', 'benefactor', 'donate', 'benevolent']


Progress: stuck at lasso/elastic net. not significant yet.